# Capstone Project: House Price Prediction

## Problem Definition Overview

* **Domain:** Real Estate / Housing
* **Problem Type:** **Regression** (predicting a continuous numeric value—the house price)
* **Problem Statement:**  
  Predict the selling price of residential properties using historical features (such as location, square footage, number of bedrooms, and amenities) to help buyers, sellers, and real estate agents accurately estimate fair market values.

* **Success Criteria:**  
  * **Evaluation Metrics:** **RMSE** (Root Mean Squared Error) and **$R^2$ Score** (aiming for $R^2 \ge 0.85$ or an acceptable error margin relative to average prices).
  * **Real-World Impact:** Reduces price estimation bias and speeds up property valuations for buyers and sellers.

* **Feasibility:** High. The dataset (`house_prices.csv`) is readily available locally in the environment.

# EDA

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

All libraries imported successfully!


In [3]:
# Load the dataset
df = pd.read_csv("house_prices.csv")

In [4]:
# 1. Inspect structure and data types

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 187531 entries, 0 to 187530
Data columns (total 21 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Index              187531 non-null  int64  
 1   Title              187531 non-null  str    
 2   Description        184508 non-null  str    
 3   Amount(in rupees)  187531 non-null  str    
 4   Price (in rupees)  169866 non-null  float64
 5   location           187531 non-null  str    
 6   Carpet Area        106858 non-null  str    
 7   Status             186916 non-null  str    
 8   Floor              180454 non-null  str    
 9   Transaction        187448 non-null  str    
 10  Furnishing         184634 non-null  str    
 11  facing             117298 non-null  str    
 12  overlooking        106095 non-null  str    
 13  Society            77853 non-null   str    
 14  Bathroom           186703 non-null  str    
 15  Balcony            138596 non-null  str    
 16  Car Parking  

In [5]:
# 2. View summary statistics

display(df.describe())

,Index,Price (in rupees),Dimensions,Plot Area
count,187531.000000,1.698660e+05,0.0,0.0
mean,93765.000000,7.583772e+03,NaN,NaN
std,54135.681003,2.724171e+04,NaN,NaN
min,0.000000,0.000000e+00,NaN,NaN
25%,46882.500000,4.297000e+03,NaN,NaN
50%,93765.000000,6.034000e+03,NaN,NaN
75%,140647.500000,9.450000e+03,NaN,NaN
max,187530.000000,6.700000e+06,NaN,NaN


In [6]:
# 3. Check first few rows

df.head()

,Index,Title,Description,Amount(in rupees),Price (in rupees),location,Carpet Area,Status,Floor,Transaction,...,facing,overlooking,Society,Bathroom,Balcony,Car Parking,Ownership,Super Area,Dimensions,Plot Area
0,0,1 BHK Ready to Occupy Flat for sale in Srushti...,"Bhiwandi, Thane has an attractive 1 BHK Flat f...",42 Lac,6000.0,thane,500 sqft,Ready to Move,10 out of 11,Resale,...,NaN,NaN,Srushti Siddhi Mangal Murti Complex,1,2,NaN,NaN,NaN,NaN,NaN
1,1,2 BHK Ready to Occupy Flat for sale in Dosti V...,One can find this stunning 2 BHK flat for sale...,98 Lac,13799.0,thane,473 sqft,Ready to Move,3 out of 22,Resale,...,East,Garden/Park,Dosti Vihar,2,NaN,1 Open,Freehold,NaN,NaN,NaN
2,2,2 BHK Ready to Occupy Flat for sale in Sunrise...,Up for immediate sale is a 2 BHK apartment in ...,1.40 Cr,17500.0,thane,779 sqft,Ready to Move,10 out of 29,Resale,...,East,Garden/Park,Sunrise by Kalpataru,2,NaN,1 Covered,Freehold,NaN,NaN,NaN
3,3,1 BHK Ready to Occupy Flat for sale Kasheli,This beautiful 1 BHK Flat is available for sal...,25 Lac,NaN,thane,530 sqft,Ready to Move,1 out of 3,Resale,...,NaN,NaN,NaN,1,1,NaN,NaN,NaN,NaN,NaN
4,4,2 BHK Ready to Occupy Flat for sale in TenX Ha...,"This lovely 2 BHK Flat in Pokhran Road, Thane ...",1.60 Cr,18824.0,thane,635 sqft,Ready to Move,20 out of 42,Resale,...,West,"Garden/Park, Main Road",TenX Habitat Raymond Realty,2,NaN,1 Covered,Co-operative Society,NaN,NaN,NaN


In [7]:
# 4. Check missing values

df.isnull().sum()

Index                     0
Title                     0
Description            3023
Amount(in rupees)         0
Price (in rupees)     17665
location                  0
Carpet Area           80673
Status                  615
Floor                  7077
Transaction              83
Furnishing             2897
facing                70233
overlooking           81436
Society              109678
Bathroom                828
Balcony               48935
Car Parking          103357
Ownership             65517
Super Area           107685
Dimensions           187531
Plot Area            187531
dtype: int64

# Preprocessing & Feature Engineering.

In [8]:


# 1. Start fresh from original df
df_clean = df.copy()

# 2. Drop 100% empty columns & uninformative identifiers
drop_cols = ['Dimensions', 'Plot Area', 'Index', 'Title', 'Description']
df_clean = df_clean.drop(columns=[c for c in drop_cols if c in df_clean.columns])

# 3. Drop rows where target variable is missing
df_clean = df_clean.dropna(subset=['Price (in rupees)'])

# 4. Extract numeric square footage safely
if 'Carpet Area' in df_clean.columns:
    df_clean['Carpet_Area_sqft'] = df_clean['Carpet Area'].astype(str).str.extract(r'(\d+\.?\d*)').astype(float)
    df_clean = df_clean.drop(columns=['Carpet Area'])

# 5. Clean numeric columns (Bathroom & Balcony)
df_clean['Bathroom'] = pd.to_numeric(df_clean['Bathroom'], errors='coerce')
df_clean['Balcony'] = pd.to_numeric(df_clean['Balcony'], errors='coerce')

# 6. Impute missing values
num_cols = ['Carpet_Area_sqft', 'Bathroom', 'Balcony']
for col in num_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

cat_cols = ['Status', 'Transaction', 'Furnishing', 'facing', 'overlooking', 'Society', 'Car Parking', 'Ownership']
for col in cat_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna('Unknown')

print("Cleaned Dataset Shape:", df_clean.shape)
print("\nRemaining Missing Values:")
print(df_clean.isnull().sum())
df_clean.head()

Cleaned Dataset Shape: (169866, 16)

Remaining Missing Values:
Amount(in rupees)        0
Price (in rupees)        0
location                 0
Status                   0
Floor                 6843
Transaction              0
Furnishing               0
facing                   0
overlooking              0
Society                  0
Bathroom                 0
Balcony                  0
Car Parking              0
Ownership                0
Super Area           93631
Carpet_Area_sqft         0
dtype: int64


,Amount(in rupees),Price (in rupees),location,Status,Floor,Transaction,Furnishing,facing,overlooking,Society,Bathroom,Balcony,Car Parking,Ownership,Super Area,Carpet_Area_sqft
0,42 Lac,6000.0,thane,Ready to Move,10 out of 11,Resale,Unfurnished,Unknown,Unknown,Srushti Siddhi Mangal Murti Complex,1.0,2.0,Unknown,Unknown,NaN,500.0
1,98 Lac,13799.0,thane,Ready to Move,3 out of 22,Resale,Semi-Furnished,East,Garden/Park,Dosti Vihar,2.0,2.0,1 Open,Freehold,NaN,473.0
2,1.40 Cr,17500.0,thane,Ready to Move,10 out of 29,Resale,Unfurnished,East,Garden/Park,Sunrise by Kalpataru,2.0,2.0,1 Covered,Freehold,NaN,779.0
4,1.60 Cr,18824.0,thane,Ready to Move,20 out of 42,Resale,Unfurnished,West,"Garden/Park, Main Road",TenX Habitat Raymond Realty,2.0,2.0,1 Covered,Co-operative Society,NaN,635.0
5,45 Lac,6618.0,thane,Ready to Move,2 out of 7,Resale,Unfurnished,East,"Garden/Park, Main Road",Virat Aangan,1.0,1.0,Unknown,Co-operative Society,680 sqft,1050.0


In [9]:
from sklearn.model_selection import train_test_split

# 1. Drop high-missing redundant column 'Super Area'
if 'Super Area' in df_clean.columns:
    df_clean = df_clean.drop(columns=['Super Area'])

# 2. Fill missing Floor values with 'Unknown'
if 'Floor' in df_clean.columns:
    df_clean['Floor'] = df_clean['Floor'].fillna('Unknown')

# 3. Drop 'Amount(in rupees)' as it is a text duplicate of the numeric 'Price (in rupees)'
if 'Amount(in rupees)' in df_clean.columns:
    df_clean = df_clean.drop(columns=['Amount(in rupees)'])

# Verify zero missing values remain
print("Remaining Missing Values Count:")
print(df_clean.isnull().sum().sum())

# 4. Separate Features (X) and Target (y)
X = df_clean.drop(columns=['Price (in rupees)'])
y = df_clean['Price (in rupees)']

# 5. One-Hot Encode categorical variables
X_encoded = pd.get_dummies(X, drop_first=True)

# 6. Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

print(f"\nFinal Shapes:")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")

Remaining Missing Values Count:
0

Final Shapes:
X_train: (135892, 10953), y_train: (135892,)
X_test:  (33974, 10953), y_test:  (33974,)


# Model Training & Evaluation Setup

We will evaluate the models using three standard regression metrics:
* **MAE (Mean Absolute Error):** Average error in absolute terms.
* **RMSE (Root Mean Squared Error):** Penalizes larger errors more heavily.
* **$R^2$ Score:** Indicates how much variance in price is explained by the model (closer to $1.0$ is better).


In [10]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [ ]:

# ============================================
# MODEL TRAINING & EVALUATION
# ============================================

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd
import time

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(
        random_state=42,
        max_depth=20
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        max_depth=20,
        random_state=42,
        n_jobs=-1
    )
}

results = []

for name, model in models.items():

    print(f"\nTraining {name}...")
    start = time.time()

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, predictions)

    results.append([
        name,
        mae,
        mse,
        rmse,
        r2
    ])

    print(f"Completed in {time.time() - start:.2f} seconds")
    print(f"MAE  : {mae:.2f}")
    print(f"RMSE : {rmse:.2f}")
    print(f"R²   : {r2:.4f}")

results_df = pd.DataFrame(
    results,
    columns=["Model", "MAE", "MSE", "RMSE", "R² Score"]
)

print("\n========== MODEL COMPARISON ==========") 
display(results_df)


Training Linear Regression...
